# Chronos-2 trading features with SB3 PPO

This notebook mirrors the Chronos trading setup from the native REINFORCE tutorial, but trains `stable_baselines3.PPO`. The install surface stays compact: `crosslearn[chronos]` covers the package functionality, while `gym-anytrading`, `pandas`, and `matplotlib` remain notebook-level dependencies.
            

In [ ]:
# Uncomment in a fresh environment:
# %pip install 'crosslearn[chronos]' gym-anytrading pandas matplotlib
            

In [ ]:
import numpy as np
import pandas as pd

from gym_anytrading.datasets import STOCKS_GOOGL
from gym_anytrading.envs import StocksEnv
from stable_baselines3 import PPO

from crosslearn.extractors import ChronosEmbedder, ChronosExtractor
            

In [3]:
LOOKBACK = 30
FEATURE_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']
SELECTED_COLUMNS = ['Close', 'Volume']
FRAME_BOUND = (LOOKBACK, len(STOCKS_GOOGL))

print(STOCKS_GOOGL.head())
print('Frame bound:', FRAME_BOUND)
print('Feature columns:', FEATURE_COLUMNS)
print('Chronos-selected columns:', SELECTED_COLUMNS)
            

                  Open        High         Low       Close   Adj Close  \
Date                                                                     
2009-05-22  198.528534  199.524521  196.196198  196.946945  196.946945   
2009-05-26  196.171173  202.702698  195.195190  202.382385  202.382385   
2009-05-27  203.023026  206.136139  202.607605  202.982986  202.982986   
2009-05-28  204.544540  206.016022  202.507507  205.405411  205.405411   
2009-05-29  206.261261  208.823822  205.555557  208.823822  208.823822   

             Volume  
Date                 
2009-05-22  3433700  
2009-05-26  6202700  
2009-05-27  6062500  
2009-05-28  5332200  
2009-05-29  5291100  
Frame bound: (30, 2335)
Feature columns: ['Open', 'High', 'Low', 'Close', 'Volume']
Chronos-selected columns: ['Close', 'Volume']


In [4]:
def online_process_data(env):
    start = env.frame_bound[0] - env.window_size
    end = env.frame_bound[1]
    prices = env.df.loc[:, 'Close'].to_numpy()[start:end]
    signal_features = env.df.loc[:, FEATURE_COLUMNS].to_numpy(dtype=np.float32)[start:end]
    return prices, signal_features


class OnlineStocksEnv(StocksEnv):
    _process_data = online_process_data


def make_online_env():
    return OnlineStocksEnv(
        df=STOCKS_GOOGL,
        window_size=LOOKBACK,
        frame_bound=FRAME_BOUND,
    )

sample_env = make_online_env()
print('Online observation space:', sample_env.observation_space)
sample_env.close()
            

Online observation space: Box(-1e+10, 1e+10, (30, 5), float32)


## Online Chronos extraction with PPO

The extractor is supplied through SB3 `policy_kwargs`. For PPO rollout, batch, and optimization details beyond this minimal setup, use the SB3 documentation as the main reference.
            

In [5]:
env = make_online_env()
model = PPO(
    'MlpPolicy',
    env,
    policy_kwargs={
        'features_extractor_class': ChronosExtractor,
        'features_extractor_kwargs': {
            'feature_names': FEATURE_COLUMNS,
            'selected_columns': SELECTED_COLUMNS,
        },
    },
    verbose=1,
    seed=42,
)
model.learn(total_timesteps=10_000) 
            

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------
| time/              |      |
|    fps             | 4    |
|    iterations      | 1    |
|    time_elapsed    | 472  |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.3e+03     |
|    ep_rew_mean          | 525         |
| time/                   |             |
|    fps                  | 1           |
|    iterations           | 2           |
|    time_elapsed         | 3529        |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007914944 |
|    clip_fraction        | 0.0281      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | 0           |
|    learning_rate        | 0.0003      |
|    loss               

## Offline Chronos embeddings

Offline mode precomputes the Chronos vectors first, then hands those vectors to PPO as plain numeric observations.
            

In [6]:
def build_offline_bundle(df, lookback, frame_bound):
    start = frame_bound[0] - lookback
    end = frame_bound[1]
    history = df.iloc[start:end].reset_index(drop=True).copy()

    embedder = ChronosEmbedder(
        feature_names=FEATURE_COLUMNS,
        selected_columns=SELECTED_COLUMNS,
    )
    embedded = embedder.transform_dataframe(
        history,
        lookback=lookback,
        columns=FEATURE_COLUMNS,
    )
    embedding_columns = [
        column for column in embedded.columns if column.startswith('chronos_')
    ]
    embedded = embedded.iloc[lookback - 1 :].reset_index(drop=True)

    return {
        'df': embedded,
        'prices': embedded.loc[:, 'Close'].to_numpy(dtype=np.float32),
        'signal_features': embedded.loc[:, embedding_columns].to_numpy(dtype=np.float32),
        'embedding_columns': embedding_columns,
    }


offline_bundle = build_offline_bundle(STOCKS_GOOGL, LOOKBACK, FRAME_BOUND)
print(offline_bundle['df'].head())
print('Offline feature matrix shape:', offline_bundle['signal_features'].shape)
print('Offline embedding columns:', offline_bundle['embedding_columns'][:5], '...')
            

         Open        High         Low       Close   Adj Close   Volume  \
0  203.453461  205.525528  201.031036  205.010010  205.010010  4520600   
1  204.324326  204.799805  198.188187  198.513519  198.513519  6512000   
2  200.200195  203.203201  199.229233  201.446442  201.446442  6875500   
3  203.263260  207.432434  203.103104  205.400406  205.400406  6544600   
4  204.994995  208.893890  204.554550  207.407410  207.407410  5847300   

   chronos_0  chronos_1  chronos_2  chronos_3  ...  chronos_758  chronos_759  \
0  -0.051934  -0.067057  -0.054964  -0.880732  ...    -0.134393     0.354705   
1  -0.051934  -0.067057  -0.054964  -0.880732  ...    -0.134393     0.354705   
2  -0.051934  -0.067057  -0.054964  -0.880732  ...    -0.134393     0.354705   
3  -0.051934  -0.067057  -0.054964  -0.880732  ...    -0.134393     0.354705   
4  -0.051934  -0.067057  -0.054964  -0.880732  ...    -0.134393     0.354705   

   chronos_760  chronos_761  chronos_762  chronos_763  chronos_764  \
0   

In [8]:
class OfflineStocksEnv(StocksEnv):
    def __init__(self, prices, signal_features, **kwargs):
        self._prices = prices
        self._signal_features = signal_features.astype(np.float32)
        super().__init__(**kwargs)

    def _process_data(self):
        return self._prices, self._signal_features


def make_offline_env():
    return OfflineStocksEnv(
        prices=offline_bundle['prices'],
        signal_features=offline_bundle['signal_features'],
        df=offline_bundle['df'],
        window_size=1,
        frame_bound=(1, len(offline_bundle['df'])),
    )

offline_env = make_offline_env()
offline_model = PPO('MlpPolicy', offline_env, verbose=1, seed=42)
offline_model.learn(total_timesteps=10_000)
            

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-----------------------------
| time/              |      |
|    fps             | 1934 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 2.3e+03      |
|    ep_rew_mean          | 563          |
| time/                   |              |
|    fps                  | 1297         |
|    iterations           | 2            |
|    time_elapsed         | 3            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0022893322 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.691       |
|    explained_variance   | 0            |
|    learning_rate        | 0.0003       |
|    los